<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/07-joins-deep-dive/00-join-types.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 7.0 — Join types: inner, outer, semi, anti, cross

Two tables from `data/`: `orders.csv` (fact, 41 rows) and the new
`customers.csv` (dimension, 21 rows). Deliberate damage: order 1037
references missing customer `c019`; customers `c021`/`c022` have no
orders. Watch what each join type does with those three keys.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

spark = (
    SparkSession.builder
    .appName("7.0-join-types")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_SCHEMA = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_date", DateType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("country", StringType(), True),
])

CUSTOMERS_SCHEMA = StructType([
    StructField("customer_id", StringType(), False),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("country", StringType(), True),
    StructField("signup_date", DateType(), True),
    StructField("segment", StringType(), True),
])

orders = spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, schema=ORDERS_SCHEMA)
customers = spark.read.csv(f"{DATA_DIR}/customers.csv", header=True, schema=CUSTOMERS_SCHEMA)

print("orders:", orders.count(), "| customers:", customers.count())

## Inner and the three outers — four `how` values, four row counts

40 / 41 / 42 / 43. All four numbers are explained by three keys:
orphan order 1037 (c019), and orderless customers c021/c022.

In [ ]:
for how in ["inner", "left", "right", "full"]:
    n = orders.join(customers, on="customer_id", how=how).count()
    print(f"{how:>6}: {n} rows")

In [ ]:
# The rows each outer join manufactures. Note the nulls were in NEITHER file —
# the join created them.
left_joined = orders.join(customers, "customer_id", "left")
left_joined.filter(col("name").isNull()).select(
    "order_id", "customer_id", "product", "name", "segment"
).show()

right_joined = orders.join(customers, "customer_id", "right")
right_joined.filter(col("order_id").isNull()).select(
    "order_id", "customer_id", "name", "city", "segment"
).show()

## Semi and anti — joins that filter

Only left-side columns come back; matches are never duplicated.
The anti join IS the referential-integrity audit.

In [ ]:
orders.join(customers, "customer_id", "left_semi").count()   # 40 — orders with a real customer

In [ ]:
# Orphan foreign keys: orders whose customer doesn't exist
orders.join(customers, "customer_id", "left_anti").show()

In [ ]:
# ...and the other direction: customers who never ordered
customers.join(orders, "customer_id", "left_anti").select(
    "customer_id", "name", "country", "signup_date"
).show()

In [ ]:
# Semi vs inner: c001 has 3 orders. Inner emits 3 rows for c001; semi emits 1.
inner_c1 = customers.join(orders, "customer_id", "inner").filter(col("customer_id") == "c001")
semi_c1 = customers.join(orders, "customer_id", "left_semi").filter(col("customer_id") == "c001")
print("inner:", inner_c1.count(), "| semi:", semi_c1.count())

## Joins multiply — not append

Join the same two tables on `country` (semantically nonsense, but
watch the arithmetic): per key, output = left rows x right rows.

In [ ]:
by_country = orders.join(customers, "country", "inner")
print("41 join 21 on country ->", by_country.count(), "rows")

# per-key breakdown: 13x6 + 8x5 + 10x5 + 7x3 = 189
orders.groupBy("country").count().withColumnRenamed("count", "n_orders").join(
    customers.groupBy("country").count().withColumnRenamed("count", "n_customers"),
    "country", "inner",
).withColumn("output_rows", col("n_orders") * col("n_customers")).show()

## Null keys never match (unless you ask)

The 3 null-country orders contributed nothing above: `NULL == NULL`
is not true in join conditions. `eqNullSafe` (SQL `<=>`) opts in.

In [ ]:
left = spark.createDataFrame([("a", 1), (None, 2)], "k STRING, v INT")
right = spark.createDataFrame([("a", 10), (None, 20)], "k STRING, w INT")

left.join(right, left["k"] == right["k"]).show()           # 1 row
left.join(right, left["k"].eqNullSafe(right["k"])).show()  # 2 rows — nulls matched

## Cross join — explicit on purpose

In [ ]:
print("cross:", orders.crossJoin(customers).count(), "rows (41 x 21)")

# a legitimate use: a complete grid to detect gaps against
all_pairs = customers.select("customer_id").crossJoin(orders.select("category").distinct())
print("customer x category grid:", all_pairs.count(), "rows")

## Exercises

1. Build the "orders enriched with customer data" DataFrame the way a
   reviewer would want it: keep ALL orders, add `name`/`segment`/home
   `country` (renamed so it can't collide), and a boolean
   `has_customer` flag derived from the join — no second pass.
2. Which customer-category pairs have NO orders? Use the `all_pairs`
   grid and an anti join. (c021/c022 should appear for every category.)
3. Predict, then verify: row count of `orders.join(orders_dup, "customer_id")`
   where `orders_dup` is orders unioned with itself. Explain the number.
4. Rewrite the two anti joins as `NOT EXISTS` in `spark.sql()` against
   temp views (lesson 4.0) and confirm identical counts.

In [ ]:
# your exercise code here